In [1]:
##############
# HOW TO RUN #
##############

# 1. Create the test Dask cluster. This will be used to auto-determine the appropriate block size
#    for your machine! 
from robustraster import dask_cluster_manager

dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="test")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [2]:
# 2. Load the local raster
from robustraster import dataset_manager
raster_path_list = ['./ndvi.tif']
local_raster = dataset_manager.RasterDataset(raster_path_list)

In [3]:
print(local_raster.dataset)

<xarray.Dataset> Size: 4GB
Dimensions:      (time: 1, y: 35260, x: 30281)
Coordinates:
  * time         (time) int64 8B 1
  * x            (x) float64 242kB -4.216e+05 -4.216e+05 ... 4.868e+05 4.868e+05
  * y            (y) float64 282kB -5.992e+05 -5.992e+05 ... 4.585e+05 4.585e+05
    spatial_ref  int64 8B 0
Data variables:
    band_1       (time, y, x) float32 4GB dask.array<chunksize=(1, 256, 512), meta=np.ndarray>
Attributes:
    AREA_OR_POINT:  Area


In [3]:
#print(local_raster.dataset.rio.crs)  # Should print the EPSG code or WKT string

EPSG:3310


In [3]:
# 3. Design your function here! 
 
# My target audience are for users who want to work with
# data frames, so pandas data frames are the only data structures 
# that I support for writing functions. If you'd prefer working 
# with xarrays (or possible other data structures), submit an
# issue and let me know!

# For this demo, we do a basic NDVI calculation.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['band_1'] + df['band_1']) / (df['band_1'])
    return df

In [ ]:
# 5. This code will auto-determine what the best block size
#    should be for your machine. This helps to ensure computations don't 
#    go over resources available and crash your application. Skip this step
#    if want to process the entire dataset as is.
from robustraster import udf_manager

user_defined_func = udf_manager.UserDefinedFunction()
user_defined_func.tune_user_function(local_raster, compute_ndvi, None)

In [ ]:
# 6. Shutdown the test cluster and recreate a Dask cluster with
#    full resources needed for the full computation.
dask_client = dask_cluster.get_dask_client
dask_client.shutdown()
dask_cluster.create_cluster(mode="full")

In [8]:
# 8. Do the full run and write the results to a geoTIFF!
result = user_defined_func.apply_user_function(local_raster, compute_ndvi)

# If you changed the parameters in step 3, you'll need to play around with the parameters
# below and change them accordingly. Otherwise, it should work as is.
ds_transposed = result.transpose('time', 'y', 'x')
ds_transposed['ndvi'].isel(time=0).rio.to_raster("ndvi_local.tif")